In [2]:
!pip install --quiet --upgrade "google-cloud-aiplatform[agent_engines,adk]"

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 105.2 MB/s eta 0:00:00
Installation complete.


In [20]:
import json
import vertexai
from google.oauth2 import service_account
from google.colab import userdata

service_account_info = json.loads(userdata.get('GCP_JSON_KEY'))
credentials = service_account.Credentials.from_service_account_info(service_account_info)

vertexai.init(
    project="voltaic-flag-359918",
    location="us-west1",
    credentials=credentials,
    staging_bucket="gs://spicer-adk"
)

In [21]:
from google.adk.agents import Agent
from google.adk.tools import google_search

search_agent = Agent(
    name="search_dude",
    model="gemini-2.5-flash",
    description=("You can search google"),
    instruction=("""You are a helpful chatbot. Try to answer questions to the best of your ability."""),
    tools=[google_search]
)

In [18]:
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(agent=search_agent)

for event in app.stream_query(
    user_id="user001",
    message="Give me news highlights in hockey.",
):
  print(event)

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:844: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Here are some of the latest news highlights in hockey:\n\n**Key Game Results and Performances:**\n*   The Columbus Blue Jackets secured an overtime victory against the New York Rangers, remarkably coming back after blowing a four-goal lead in the third period. Kirill Marchenko led the Blue Jackets with three points, including the game-winning goal.\n*   The Philadelphia Flyers recovered to defeat the struggling Maple Leafs in a shootout.\n*   The Detroit Red Wings rallied to beat the Predators, though they lost Gibson to an upper-body injury during the game.\n*   The Dallas Stars are on an impressive run, scoring six straight goals to defeat the Canucks and achieve a franchise-record ninth consecutive win.\n*   The Seattle Kraken ended the Hurricanes\' 12-game point streak, with Daccord making 35 saves.\n*   The Avalanche spoiled D.J. Smith\'s debut as the Kings\' coach.\n\n**Coaching Changes and Player News:**\n*   

In [23]:
from vertexai import agent_engines
remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"]
)

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.12.3', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.139.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai.agent_engines:Using bucket spicer-adk
INFO:vertexai.agent_engines:Wrote to gs://spicer-adk/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://spicer-adk/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://spicer-adk/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/380137098345/locations/us-west1/reasoningEngines/3008738802618335232/operations/72059669

In [25]:
for event in remote_agent.stream_query(
    user_id="agent_engine_test",
    message="Give me news highlights in hockey.",
):
  print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'Here are some of the latest news highlights in hockey:\n\n**Recent Game Results and Noteworthy Performances:**\n*   The Columbus Blue Jackets secured an overtime victory against the New York Rangers after initially blowing a four-goal lead in the third period. Kirill Marchenko scored twice for the Blue Jackets, while Gabe Perrault had two goals for the Rangers.\n*   The Philadelphia Flyers recovered to defeat the struggling Toronto Maple Leafs in a shootout.\n*   The Dallas Stars achieved a franchise-record ninth consecutive win by scoring six straight goals against the Vancouver Canucks.\n*   The Seattle Kraken, with a 35-save performance from Daccord, ended the Carolina Hurricanes\' 12-game point streak.\n*   The Detroit Red Wings rallied past the Nashville Predators, with Johansson scoring a short-handed goal, though the Red Wings\' Gibson left the game with an upper-body injury.\n*   The Colorado Avalanche spoile